# 24 — Heston Stochastic Volatility

- **Semi-analytic** (Fourier integration, Lord-Kahl)
- **COS method** (Fang-Oosterlee)
- **Gatheral expansion** (fast approximation)
- **Monte Carlo** (Andersen QE scheme)
- **Finite-difference** (2D ADI)
- **Calibration** to market prices
- Implied vol surface
- Bates (Heston + jumps)
- AD: model Greeks

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.heston import heston_price, heston_price_cos
from ql_jax.engines.analytic.heston_variants import heston_expansion_price, bates_price
from ql_jax.engines.mc.heston import mc_heston_price
from ql_jax.engines.fd.heston import fd_heston_price
from ql_jax.termstructures.volatility.heston_vol import heston_implied_vol, build_heston_vol_surface
from ql_jax.models.equity.heston import calibrate_heston

S, K, T = 100.0, 100.0, 1.0
r, q = 0.03, 0.01
v0, kappa, theta, xi, rho = 0.04, 1.5, 0.04, 0.3, -0.7

---
## 1. Multi-Engine Pricing

In [ ]:
p_fourier = float(heston_price(S, K, T, r, q, v0, kappa, theta, xi, rho, 1))
p_cos     = float(heston_price_cos(S, K, T, r, q, v0, kappa, theta, xi, rho, 1))
p_exp     = float(heston_expansion_price(S, K, T, r, q, v0, kappa, theta, xi, rho, option_type=1))
p_fd      = float(fd_heston_price(S, K, T, r, q, v0, kappa, theta, xi, rho, 1, n_x=150, n_v=80, n_t=200))

mc_val, mc_se = mc_heston_price(S, K, T, r, q, v0, kappa, theta, xi, rho, 1,
                                 n_paths=200_000, n_steps=200, scheme='qe')
p_mc, se_mc = float(mc_val), float(mc_se)

# QL reference
today = QL.Date(15, 6, 2024)
QL.Settings.instance().evaluationDate = today
dc = QL.Actual365Fixed()
spot_h = QL.QuoteHandle(QL.SimpleQuote(S))
r_h = QL.YieldTermStructureHandle(QL.FlatForward(today, r, dc))
q_h = QL.YieldTermStructureHandle(QL.FlatForward(today, q, dc))
proc = QL.HestonProcess(r_h, q_h, spot_h, v0, kappa, theta, xi, rho)
model = QL.HestonModel(proc)
engine = QL.AnalyticHestonEngine(model)
opt = QL.VanillaOption(QL.PlainVanillaPayoff(QL.Option.Call, K),
                        QL.EuropeanExercise(today + QL.Period(1, QL.Years)))
opt.setPricingEngine(engine)
p_ql = opt.NPV()

print(f"{'Engine':<20s} {'Price':>10s} {'vs QL':>10s}")
print("-" * 45)
print(f"{'QL analytic':<20s} {p_ql:>10.6f}")
print(f"{'Fourier':<20s} {p_fourier:>10.6f} {abs(p_fourier-p_ql):>10.2e}")
print(f"{'COS':<20s} {p_cos:>10.6f} {abs(p_cos-p_ql):>10.2e}")
print(f"{'Expansion':<20s} {p_exp:>10.6f} {abs(p_exp-p_ql):>10.2e}")
print(f"{'FD ADI':<20s} {p_fd:>10.6f} {abs(p_fd-p_ql):>10.2e}")
print(f"{'MC QE (200k)':<20s} {p_mc:>10.6f} {abs(p_mc-p_ql):>10.2e}")

---
## 2. Implied Vol Surface

In [ ]:
strikes = jnp.linspace(70, 130, 25)
maturities = jnp.array([0.25, 0.5, 1.0, 2.0, 3.0])

vol_surface = build_heston_vol_surface(S, r, q, v0, kappa, theta, xi, rho, strikes, maturities)

fig, ax = plt.subplots(figsize=(10, 6))
for i, m in enumerate(maturities):
    ax.plot(np.array(strikes), np.array(vol_surface[i]) * 100, label=f'{float(m):.2f}Y')

ax.set_xlabel('Strike')
ax.set_ylabel('Implied Vol (%)')
ax.set_title('Heston Implied Vol Surface')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 3. Bates Model (Heston + Jumps)

In [ ]:
p_bates = float(bates_price(S, K, T, r, q, v0, kappa, theta, xi, rho,
                             jump_intensity=0.5, jump_mean=-0.1, jump_vol=0.15,
                             option_type=1))

print(f"Heston call     : {p_fourier:.6f}")
print(f"Bates call      : {p_bates:.6f}")
print(f"Jump component  : {p_bates - p_fourier:.4f}")

---
## 4. AD: Model Greeks

In [ ]:
def heston_call(spot, v0_p, kappa_p, theta_p, xi_p, rho_p):
    return heston_price(spot, K, T, r, q, v0_p, kappa_p, theta_p, xi_p, rho_p, 1)

grad_all = jax.grad(heston_call, argnums=(0, 1, 2, 3, 4, 5))
greeks = grad_all(S, v0, kappa, theta, xi, rho)

labels = ['Delta (dV/dS)', 'dV/dv0', 'dV/dκ', 'dV/dθ', 'dV/dξ', 'dV/dρ']
for lbl, g in zip(labels, greeks):
    print(f"  {lbl:<15s} : {float(g):12.6f}")

---
## 5. Calibration

In [ ]:
# Generate "market" prices from known params
cal_strikes = jnp.array([80, 90, 95, 100, 105, 110, 120], dtype=jnp.float64)
cal_mats    = jnp.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
market_prices = jnp.array([float(heston_price(S, float(k), 0.5, r, q, v0, kappa, theta, xi, rho, 1))
                           for k in cal_strikes])

result = calibrate_heston(
    S, r, q, cal_strikes, cal_mats, market_prices,
    initial_params=(0.06, 2.0, 0.05, 0.4, -0.5), max_iter=100)

print("Calibrated parameters:")
print(f"  v0    : {result.params[0]:.6f}  (true: {v0})")
print(f"  kappa : {result.params[1]:.6f}  (true: {kappa})")
print(f"  theta : {result.params[2]:.6f}  (true: {theta})")
print(f"  xi    : {result.params[3]:.6f}  (true: {xi})")
print(f"  rho   : {result.params[4]:.6f}  (true: {rho})")

---
## 6. Exercises

1. **Vol-of-vol sensitivity**: Plot the smile shape for ξ = 0.1, 0.3, 0.5, 1.0.
2. **Term structure of skew**: Plot ATM skew ($\partial \sigma_{\text{imp}} / \partial K|_{K=S}$) vs maturity using AD.
3. **Feller condition**: Check $2\kappa\theta > \xi^2$ and see what happens to MC when violated.

---
*End of Notebook 24*